# Chemical Composition (BWMD): from data entry to RDF

This notebook shows how to describe the chemical composition of a material
and convert that description into a standardised, machine-readable RDF graph
following the
[BWMD Ontology](https://gitlab.cc-asp.fraunhofer.de/EMI_datamanagement/bwmd_ontology).

**You only need to edit one file:** `docs/example.input.json`.  Everything
else is automatic.

---

## What the notebook does

```
example.input.json
  │  your material name and element fractions
  │
  ▼
Transform
  │  converts your plain JSON into a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms the graph is structurally correct
```

---

## Environment setup

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

In [1]:
%pip install -q jsonata-python rdflib pyshacl pyyaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, json, pathlib, yaml, pyshacl, rdflib
from jsonata.jsonata import Jsonata
from importlib.metadata import version

print("Python        :", sys.executable)
print("jsonata-python", version("jsonata-python"))
print("rdflib        ", version("rdflib"))
print("pyshacl       ", version("pyshacl"))

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # BWMD/

Python        : /root/semantic-dataspace/.venv/bin/python3
jsonata-python 0.6.2
rdflib         7.6.0
pyshacl        0.31.0


---
## Step 1: Describe your material

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `material_name` | yes | A name or identifier for the material |
| `elements` | yes | List of `{symbol, value, unit}`, one per element |
| `material_id` | no | Custom IRI slug (auto-derived from `material_name` if omitted) |
| `comp_id` | no | Custom IRI slug for the composition node (auto-derived if omitted) |

Element `unit` must be `"mass%"`, `"vol%"`, or `"mol%"`.  All elements in one
composition should use the same unit.

In [3]:
simplified_path = HERE / "example.input.json"
simplified = json.loads(simplified_path.read_text())

print(json.dumps(simplified, indent=2))

{
  "label": "316L Chemical Composition",
  "base_element": {
    "symbol": "Fe"
  },
  "fractions": [
    {
      "symbol": "Cr",
      "min_value": 16.0,
      "max_value": 18.0,
      "unit": "wt.%"
    },
    {
      "symbol": "Ni",
      "min_value": 10.0,
      "max_value": 14.0,
      "unit": "wt.%"
    },
    {
      "symbol": "Mo",
      "min_value": 2.0,
      "max_value": 3.0,
      "unit": "wt.%"
    },
    {
      "symbol": "Mn",
      "min_value": 0.0,
      "max_value": 2.0,
      "unit": "wt.%"
    },
    {
      "symbol": "Si",
      "min_value": 0.0,
      "max_value": 0.75,
      "unit": "wt.%"
    },
    {
      "symbol": "C",
      "min_value": 0.0,
      "max_value": 0.03,
      "unit": "wt.%"
    },
    {
      "symbol": "P",
      "min_value": 0.0,
      "max_value": 0.045,
      "unit": "wt.%"
    },
    {
      "symbol": "S",
      "min_value": 0.0,
      "max_value": 0.03,
      "unit": "wt.%"
    }
  ]
}


---
## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document, the intermediate
format that carries both the data and its ontology mapping.

Node identifiers (`material_id`, `comp_id`) are derived automatically from
`material_name` as a URL-safe slug (e.g. `"316L Stainless Steel"` →
`mat-316l-stainless-steel`).  Override them in the input only when you need
a specific IRI to match an existing knowledge graph node.

You can also run the transform from the command line:
```
python -m jsonata -e ../simplified/transform.jsonata -i example.input.json
```

In [4]:
expr = (SCHEMA / "simplified" / "transform.jsonata").read_text()
oold_doc = Jsonata(expr).evaluate(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/chemical-composition/BWMD/#v1.0.0",
  "type": "bwmd:ChemicalComposition",
  "label": "316L Chemical Composition",
  "base_element": {
    "type": "bwmd:BaseElementOfComposition",
    "element_symbol": "Fe"
  },
  "fractions": [
    {
      "type": "bwmd:WeightFraction",
      "element_symbol": "Cr",
      "min_value": 16.0,
      "max_value": 18.0,
      "unit_symbol": "wt.%"
    },
    {
      "type": "bwmd:WeightFraction",
      "element_symbol": "Ni",
      "min_value": 10.0,
      "max_value": 14.0,
      "unit_symbol": "wt.%"
    },
    {
      "type": "bwmd:WeightFraction",
      "element_symbol": "Mo",
      "min_value": 2.0,
      "max_value": 3.0,
      "unit_symbol": "wt.%"
    },
    {
      "type": "bwmd:WeightFraction",
      "element_symbol": "Mn",
      "min_value": 0.0,
      "max_value": 2.0,
      "unit_symbol": "wt.%"
    },
    {
      "type": "bwmd:WeightFraction",
      "e

---
## Step 3: Convert to RDF

The OO-LD document is parsed as JSON-LD using the ontology context from
`specs/schema.oold.yaml`, which maps every field name to its ontology IRI.
rdflib ≥ 7 handles JSON-LD 1.1 natively.

In [5]:
context = yaml.safe_load((SCHEMA / "specs" / "schema.oold.yaml").read_text())["@context"]

g = rdflib.Dataset()
g.parse(data=json.dumps({"@context": context, **oold_doc}), format="json-ld")

print(f"Graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

Graph contains 54 triples.

@prefix bwmd: <https://www.iwm.fraunhofer.de/ontologies/bwmd-ontology#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

[] a bwmd:ChemicalComposition ;
    rdfs:label "316L Chemical Composition" ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/chemical-composition/BWMD/#v1.0.0> ;
    bwmd:hasPart [ a bwmd:WeightFraction ;
            bwmd:hasDoubleLiteralMaxValue 4.5e-02 ;
            bwmd:hasDoubleLiteralMinValue 0e+00 ;
            bwmd:hasUnitSymbol "wt.%" ;
            bwmd:refersToElementSymbol "P" ],
        [ a bwmd:WeightFraction ;
            bwmd:hasDoubleLiteralMaxValue 2e+00 ;
            bwmd:hasDoubleLiteralMinValue 0e+00 ;
            bwmd:hasUnitSymbol "wt.%" ;
            bwmd:refersToElementSymbol "Mn" ],
        [ a bwmd:WeightFraction ;
            bwmd:hasDoubleLiteralMaxValue 3e+0

In [6]:
# Optional: save to file
out_ttl = HERE / "output_316L.ttl"
out_ttl.write_text(g.serialize(format="turtle"))
print(f"Written to {out_ttl}")

Written to /root/semantic-dataspace/semantic-schemas/schemas/chemical-composition/BWMD/docs/output_316L.ttl


---
## Step 4: Validate against the SHACL shape

The SHACL shape (`specs/shape.ttl`) checks that the graph is structurally
correct; for example, that every element fraction has a numeric value
in \[0, 100\] and an allowed unit string.

> No `inference="rdfs"` flag is needed here; all shapes target classes
> directly without relying on subclass reasoning.

In [7]:
shapes_graph = rdflib.Graph().parse(str(SCHEMA / "specs" / "shape.ttl"))

conforms, results_graph, _ = pyshacl.validate(
    g,
    shacl_graph = shapes_graph,
)

print(f"Conforms: {conforms}")
if not conforms:
    SH = rdflib.Namespace("http://www.w3.org/ns/shacl#")
    for result in results_graph.subjects(rdflib.RDF.type, SH.ValidationResult):
        msg  = results_graph.value(result, SH.resultMessage)
        path = results_graph.value(result, SH.resultPath)
        loc  = str(path).rsplit("#", 1)[-1].rsplit("/", 1)[-1] if path else None
        print(f"\n  ✗ {msg}" + (f"\n    property: {loc}" if loc else ""))

Conforms: True


---
## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the material name and element fractions |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against the SHACL shape |

To use your own material, edit `docs/example.input.json` and re-run all cells.

---

## Further reading

- [OO-LD primer](../../../docs/oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../docs/schema-format.md): how to write your own schema
- [Simplified input guide](../../../docs/simplified-input-guide.md): end-to-end workflow